In [ ]:
from pathlib import Path
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Trỏ thẳng vào thư mục data bên trong project
BASE_DATA_DIR = PROJECT_ROOT / "data" / "Dataset_Capstone"

# --- CHỌN BỘ DỮ LIỆU ĐỂ TRAIN TẠI ĐÂY ---
# Mở comment dòng bạn muốn train, comment dòng còn lại:

# DATA_DIR = BASE_DATA_DIR / "Livestock_Skin"  # Đang chọn train Lợn + Bò (Ngoài da)
DATA_DIR = BASE_DATA_DIR / "Poultry_Feces"   # Đang chọn train Gà (Phân)

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print(f"Đang nạp dữ liệu từ: {DATA_DIR.name}")
print("Thư mục Data tồn tại:", DATA_DIR.exists())
print("Torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

TRAIN_MODE = True

Project root: c:\Users\MangOS\livestock-diseases-ai
Đang nạp dữ liệu từ: Livestock_Skin
Thư mục Data tồn tại: True
Torch: 2.2.2+cu121
Device: cuda


In [2]:
from src.utils import set_seed
from src.dataset import load_dataset, make_loaders
from src.model import build_model, count_parameters, unfreeze_backbone, load_checkpoint
from src.train import train_model
from src.evaluate import run_full_evaluation, plot_training_curves

set_seed(42)
print("Project modules imported successfully.")

Project modules imported successfully.


In [3]:
# Tự động đọc data và weights từ dataset.py mới
train_dataset, val_dataset, test_dataset, info = load_dataset(DATA_DIR)

train_loader, val_loader, test_loader = make_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32, # Giữ ở mức 32 để tránh tràn RAM
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

print("Dataset loaded successfully.")
print("Split sizes:", info["split_sizes"])
print("Number of classes:", info["n_classes"])
print("Trọng số phân lớp (Class Weights):", info["class_weights"])

Dataset loaded successfully.
Split sizes: {'train': 2169, 'val': 3150, 'test': 1680, 'total': 6999}
Number of classes: 10
Trọng số phân lớp (Class Weights): tensor([0.9683, 0.9904, 0.9683, 1.0136, 0.9904, 1.0088, 1.1356, 0.9152, 0.9859,
        1.0529])


In [4]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Min pixel value:", images.min().item())
print("Max pixel value:", images.max().item())
print("First 10 labels:", labels[:10].tolist())

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Min pixel value: -2.1179039478302
Max pixel value: 2.640000104904175
First 10 labels: [5, 4, 7, 2, 5, 7, 5, 0, 1, 5]


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tự động scale số lớp theo n_classes của dataset
model = build_model(
    n_classes=info["n_classes"],
    freeze_backbone=True,
).to(device)

params = count_parameters(model)

print("Device:", device)
print("Model device:", next(model.parameters()).device)
print("Model created successfully.")
print("Parameters:", params)

Device: cuda
Model device: cuda:0
Model created successfully.
Parameters: {'total': 11181642, 'trainable': 5130, 'frozen': 11176512}


In [6]:
with torch.no_grad():
    sample_outputs = model(images.to(device))

print("Sample output shape:", sample_outputs.shape)

Sample output shape: torch.Size([32, 10])


In [ ]:
phase1_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=10,
    learning_rate=0.001,
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_chicken_phase1_best.pth"),
    history_path=str(RESULTS_DIR / "history_chicken_phase1.json"),
    phase_name="phase1_frozen_backbone",
)

print("Phase 1 training completed.")


Training phase1_frozen_backbone
Epochs: 10
Learning rate: 0.001
Trainable parameters: 5,130


Epoch 01/10 | Train Loss: 2.8158 | Train Acc: 0.1692 | Val Loss: 2.0329 | Val Acc: 0.2959 | Time: 20s
  --> Best checkpoint saved! (Val Acc: 0.2959)


Epoch 02/10 | Train Loss: 2.1120 | Train Acc: 0.3084 | Val Loss: 1.8014 | Val Acc: 0.3933 | Time: 9s
  --> Best checkpoint saved! (Val Acc: 0.3933)


Epoch 03/10 | Train Loss: 1.8565 | Train Acc: 0.3794 | Val Loss: 1.5574 | Val Acc: 0.4835 | Time: 9s
  --> Best checkpoint saved! (Val Acc: 0.4835)


Epoch 04/10 | Train Loss: 1.7273 | Train Acc: 0.4025 | Val Loss: 1.4496 | Val Acc: 0.5137 | Time: 9s
  --> Best checkpoint saved! (Val Acc: 0.5137)


...

Epoch 06/10 | Train Loss: 1.5916 | Train Acc: 0.4647 | Val Loss: 1.3308 | Val Acc: 0.5543 | Time: 9s
  --> Best checkpoint saved! (Val Acc: 0.5543)


Epoch 07/10 | Train Loss: 1.5261 | Train Acc: 0.4952 | Val Loss: 1.3309 | Val Acc: 0.5521 | Time: 9s


Epoch 08/10 | Train Loss: 1.4984 | Train Acc: 0.4809 | Val Loss: 1.3270 | Val Acc: 0.5571 | Time: 9s
  --> Best checkpoint saved! (Val Acc: 0.5571)


Epoch 09/10 | Train Loss: 1.4843 | Train Acc: 0.4979 | Val Loss: 1.3233 | Val Acc: 0.5527 | Time: 9s


Epoch 10/10 | Train Loss: 1.5217 | Train Acc: 0.4809 | Val Loss: 1.3138 | Val Acc: 0.5581 | Time: 9s
  --> Best checkpoint saved! (Val Acc: 0.5581)
Phase 1 training completed.


In [ ]:
phase1_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase1",
)

plot_training_curves(
    phase1_history,
    save_path=FIGURES_DIR / "training2_curves_phase1.png",
)
print("Phase 1 evaluation and plotting completed.")


--- Evaluation Results (phase1) ---
Accuracy   : 0.5589
Macro F1   : 0.5412
Weighted F1: 0.5442
Phase 1 evaluation and plotting completed.


In [ ]:
load_checkpoint(
    model,
    str(MODELS_DIR / "resnet18_chicken_phase1_best.pth"),
)

unfreeze_backbone(model)

params = count_parameters(model)

print("Best Phase 1 checkpoint loaded.")
print("Backbone unfrozen for Phase 2.")
print("Parameters:", params)

Best Phase 1 checkpoint loaded.
Backbone unfrozen for Phase 2.
Parameters: {'total': 11181642, 'trainable': 11181642, 'frozen': 0}


In [ ]:
phase2_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=15,
    learning_rate=0.0001, # LR nhỏ để tinh chỉnh mượt mà
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_chicken_phase2_best.pth"),
    history_path=str(RESULTS_DIR / "history_chicken_phase2.json"),
    phase_name="phase2_full_finetuning",
)

print("Phase 2 training completed.")


Training phase2_full_finetuning
Epochs: 15
Learning rate: 0.0001
Trainable parameters: 11,181,642


Epoch 01/15 | Train Loss: 1.3078 | Train Acc: 0.5703 | Val Loss: 0.9683 | Val Acc: 0.6848 | Time: 16s
  --> Best checkpoint saved! (Val Acc: 0.6848)


Epoch 02/15 | Train Loss: 0.8857 | Train Acc: 0.6893 | Val Loss: 0.9296 | Val Acc: 0.6978 | Time: 16s
  --> Best checkpoint saved! (Val Acc: 0.6978)


Epoch 03/15 | Train Loss: 0.6390 | Train Acc: 0.7755 | Val Loss: 0.9372 | Val Acc: 0.7152 | Time: 16s
  --> Best checkpoint saved! (Val Acc: 0.7152)


Epoch 04/15 | Train Loss: 0.5342 | Train Acc: 0.8105 | Val Loss: 1.0083 | Val Acc: 0.7143 | Time: 16s


...

Epoch 11/15 | Train Loss: 0.1355 | Train Acc: 0.9590 | Val Loss: 0.9018 | Val Acc: 0.7610 | Time: 16s


Epoch 12/15 | Train Loss: 0.1183 | Train Acc: 0.9663 | Val Loss: 0.9002 | Val Acc: 0.7603 | Time: 16s


Epoch 13/15 | Train Loss: 0.1139 | Train Acc: 0.9700 | Val Loss: 0.9110 | Val Acc: 0.7600 | Time: 16s


Epoch 14/15 | Train Loss: 0.1076 | Train Acc: 0.9700 | Val Loss: 0.9066 | Val Acc: 0.7568 | Time: 16s


Epoch 15/15 | Train Loss: 0.1046 | Train Acc: 0.9696 | Val Loss: 0.9182 | Val Acc: 0.7537 | Time: 16s
Phase 2 training completed.


In [ ]:
phase2_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase2",
)

plot_training_curves(
    phase2_history,
    save_path=FIGURES_DIR / "training2_curves_phase2.png",
)
print("Phase 2 evaluation completed.")


--- Evaluation Results (phase2) ---
Accuracy   : 0.7619
Macro F1   : 0.7577
Weighted F1: 0.7600
Phase 2 evaluation completed.
